# 01.7 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/01-working-with-data/07-plotting/exercises.ipynb)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
sns.set_theme(style="whitegrid")

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, pandas as pd
sys.path.insert(0, str(data_path("download.py").parent))
from download import ensure_online_retail

raw = pd.read_excel(ensure_online_retail(), sheet_name="Year 2010-2011")
print(raw.shape)
raw.head()

In [ ]:
def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df.copy()
    out = out[~out["Invoice"].astype(str).str.startswith("C")]   # drop cancellations
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)]         # drop returns / free items
    out = out.dropna(subset=["Customer ID"])                       # drop guest checkouts
    out = out.drop_duplicates()
    out["Customer ID"] = out["Customer ID"].astype(int)
    out["Revenue"] = out["Quantity"] * out["Price"]
    return out

sales = clean_retail(raw)
print(f"{len(raw):,} rows → {len(sales):,} after cleaning")

In [ ]:
sales["month"] = sales["InvoiceDate"].dt.to_period("M").dt.to_timestamp()
monthly = sales.groupby("month")["Revenue"].sum()
by_country = sales.groupby("Country")["Revenue"].sum().sort_values(ascending=False)
order_values = sales.groupby("Invoice")["Revenue"].sum()
customers = sales.groupby("Customer ID").agg(
    orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
    products=("StockCode", "nunique"), items=("Quantity", "sum"),
    first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
customers["days_active"] = (customers["last_order"] - customers["first_order"]).dt.days
customers["avg_order"] = customers["spend"] / customers["orders"]

## Exercise 1: top-5 non-UK countries over time

One line per country, with a legend. Which country is growing fastest through 2011?

In [ ]:
# your code here

## Exercise 2: quantity per line, linear vs log

Histogram of `sales["Quantity"]` with 50 bins; then with `log_scale=True`. Why is the second more useful?

In [ ]:
# your code here

## Exercise 3: customer spend per country

Box plot of per-customer `spend` for the six biggest countries (log y). Which has the widest spread? Are its outliers errors?

In [ ]:
# your code here

## Exercise 4: orders vs products

Scatter `orders` vs `products` on the customer table with `alpha=0.3`. What shape, and what does it mean?

In [ ]:
# your code here